# Practical 5: Generative Adversarial Networks (GAN) Architecture and Training

## Aim
To study the architecture and training process of Generative Adversarial Networks using PyTorch.

In [ ]:
%pip install -q torch torchvision matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

## DCGAN Components

In [ ]:
latent_dim = 100

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(True),
            nn.Linear(256, 512),
            nn.ReLU(True),
            nn.Linear(512, 784),
            nn.Tanh()
        )

    def forward(self, z):
        x = self.net(z)
        return x.view(-1, 1, 28, 28)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

G = Generator().to(device)
D = Discriminator().to(device)

print('Generator and Discriminator initialized.')

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.MNIST(root='data', train=True, download=True, transform=transform)
loader = DataLoader(dataset, batch_size=128, shuffle=True)

criterion = nn.BCELoss()
opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

print('DataLoader and optimizers ready.')

## Training Loop

In [ ]:
epochs = 1
max_steps = 300
g_losses, d_losses = [], []

step = 0
for epoch in range(epochs):
    for real_imgs, _ in loader:
        real_imgs = real_imgs.to(device)
        bsz = real_imgs.size(0)

        real_labels = torch.ones(bsz, 1, device=device)
        fake_labels = torch.zeros(bsz, 1, device=device)

        # Train Discriminator
        z = torch.randn(bsz, latent_dim, device=device)
        fake_imgs = G(z).detach()

        d_real = D(real_imgs)
        d_fake = D(fake_imgs)

        loss_D = criterion(d_real, real_labels) + criterion(d_fake, fake_labels)
        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # Train Generator
        z = torch.randn(bsz, latent_dim, device=device)
        gen_imgs = G(z)
        d_gen = D(gen_imgs)

        loss_G = criterion(d_gen, real_labels)
        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        g_losses.append(loss_G.item())
        d_losses.append(loss_D.item())

        step += 1
        if step % 50 == 0:
            print(f'Step {step}: D_loss={loss_D.item():.4f}, G_loss={loss_G.item():.4f}')

        if step >= max_steps:
            break

    if step >= max_steps:
        break

print('Training completed.')

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(d_losses, label='Discriminator Loss')
plt.plot(g_losses, label='Generator Loss')
plt.title('GAN Training Loss Curves')
plt.xlabel('Training Step')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
with torch.no_grad():
    z = torch.randn(16, latent_dim, device=device)
    samples = G(z).cpu()

fig, axes = plt.subplots(4, 4, figsize=(6, 6))
idx = 0
for r in range(4):
    for c in range(4):
        axes[r, c].imshow(samples[idx, 0], cmap='gray')
        axes[r, c].axis('off')
        idx += 1
plt.suptitle('Generated Images from GAN')
plt.tight_layout()
plt.show()

## Result
The GAN architecture and adversarial training process were studied successfully using PyTorch, and synthetic images were generated.